In [1]:
import torch
import os 
import json
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import CIFAR100
from torchvision import transforms, models

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
transform_train_vit = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

transform_test_vit = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 2. Khởi tạo Dataset gốc với transform cho ViT
train_dataset_vit = CIFAR100(root="data", train=True, download=True, transform=transform_train_vit)
val_dataset_vit_clean = CIFAR100(root="data", train=True, download=False, transform=transform_test_vit)
test_dataset_vit = CIFAR100(root="data", train=False, download=True, transform=transform_test_vit)

# 3. Sử dụng đúng chỉ số phân chia (90% train, 10% val) đã cố định seed=42 từ trước
total_samples = len(train_dataset_vit)
train_size = int(0.9 * total_samples)

generator = torch.Generator().manual_seed(42)
permuted_indices = torch.randperm(total_samples, generator=generator).tolist()

train_indices = permuted_indices[:train_size]
val_indices = permuted_indices[train_size:]

# 4. Tạo Subset và DataLoader
TRAIN_SUBSET_VIT = Subset(train_dataset_vit, train_indices)
VAL_SUBSET_VIT = Subset(val_dataset_vit_clean, val_indices)

BATCH_SIZE = 64  # ViT tốn nhiều VRAM hơn ResNet, nên giảm batch size xuống 64 (hoặc 32 nếu GPU yếu)

train_loader = DataLoader(TRAIN_SUBSET_VIT, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(VAL_SUBSET_VIT, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset_vit, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)